In [1]:
from langgraph.graph import StateGraph

In [2]:
def is_dangerous_action(action: str) -> bool:
    dangerous_actions = [
        "send_email",
        "create_calendar_invite",
        "delete",
        "update",
    ]
    return action in dangerous_actions

In [3]:
def agent_node(state: dict) -> dict:
    email = state["email"]

    # Example agent decision logic
    if "meeting" in email.lower():
        action = "create_calendar_invite"
        response = "I can create a calendar invite for this meeting."
    else:
        action = "respond"
        response = "Here is a safe response to the email."

    state["action"] = action
    state["response"] = response
    state["paused"] = False
    return state

In [4]:
def hitl_checkpoint(state: dict) -> dict:
    action = state.get("action")

    if is_dangerous_action(action):
        state["paused"] = True
        print(f"Human approval required for action: {action}")
    else:
        state["paused"] = False

    return state

In [5]:
def human_decision(state: dict, decision: str) -> dict:
    if decision.lower() == "approve":
        state["paused"] = False
        print("Human approved the action.")
    else:
        state["action"] = "deny"
        state["response"] = "Action denied by human."
        state["paused"] = False
        print("Human denied the action.")

    return state

In [6]:
graph = StateGraph(dict)

graph.add_node("agent", agent_node)
graph.add_node("hitl", hitl_checkpoint)

graph.set_entry_point("agent")
graph.add_edge("agent", "hitl")

app = graph.compile()

In [7]:
state = {
    "email": "Please schedule a meeting tomorrow at 10 AM"
}

out = app.invoke(state)
out

Human approval required for action: create_calendar_invite


{'email': 'Please schedule a meeting tomorrow at 10 AM',
 'action': 'create_calendar_invite',
 'response': 'I can create a calendar invite for this meeting.',
 'paused': True}

In [8]:
if out["paused"]:
    out = human_decision(out, "approve")

out

Human approved the action.


{'email': 'Please schedule a meeting tomorrow at 10 AM',
 'action': 'create_calendar_invite',
 'response': 'I can create a calendar invite for this meeting.',
 'paused': False}